# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aleezafatima-21/Aleeza-flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I am comparing three models — logistic regression, decision tree, and random forest. Logistic regression provides an interpretable linear model, while the decision tree provides a simple rule-based comparison. Random forest adds more flexibility by combining multiple decision trees and can capture nonlinear relationships between the features and the likelihood of an impressions decline.

Using several models allows me to compare increasing model complexity rather than selecting a model without evidence. Because the purpose of this lane is to prioritize pages for review, I will pay particular attention to Precision@50, which measures how many of the 50 highest-ranked pages are actually declining. I will also compare ROC-AUC, average precision, precision, recall, and F1 to give a broader view of model performance.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I used a client-grouped 80/20 train/test split. Instead of randomly splitting individual content rows, I held out approximately 20% of the clients entirely for testing. This prevents content from the same client appearing in both the training and test sets and reduces the risk of client-level information leaking across the split.

The final labeled dataset contained 331,436 content items across 55 clients. The training set contained 44 clients and 300,879 rows, while the held-out test set contained 11 clients and 30,557 rows. The client overlap between training and test was 0.

The target was defined as whether total GSC impressions declined from March 2026 to April 2026. There were 111,967 declining pages and 219,469 non-declining pages, giving an overall decline rate of approximately 33.8%.

Missing feature values were handled using medians calculated from the training set only. The 1,434 literal zero values in `gsc_avg_position` were treated as missing because a position of zero does not represent a valid search position. The same training-derived preprocessing was then applied to the test set.

I also sorted the merged dataset by `content_hash_id` before the split so that the row order was deterministic across repeated runs. This supports reproducibility when the same random seed and preprocessing are used.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I compared the three supervised models against a simple warehouse baseline using the same held-out client split and the same evaluation metrics.

The warehouse baseline used the training-set median of GSC impressions and the training-set median CTR as thresholds. The CTR median was 0.0 because at least half of the training rows had zero clicks. Therefore, its CTR condition effectively identifies pages with zero clicks rather than providing a graded measure of weak click-through performance.

The three supervised models were logistic regression, decision tree, and random forest. Logistic regression was evaluated using log-transformed count features, training-set median imputation, and standardization because the count features were highly right-skewed. The decision tree and random forest used the same underlying features and training/test split.

The primary ranking metric is Precision@50 because the purpose of the lane is to prioritize a small number of pages for human review. ROC-AUC, average precision, precision, recall, and F1 were also used to provide a broader comparison.

The random forest achieved the strongest overall results. It had the highest ROC-AUC, average precision, Precision@50, and F1 among the models evaluated. Therefore, it was selected as the leading candidate for the final content-opportunity ranking model.


In [1]:
# ML-08: Build warehouse feature frame and March-April decline label

import os
import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

# Connect to the FlyRank warehouse
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

con.execute(
    "CREATE OR REPLACE SECRET hf_secret "
    "(TYPE HUGGINGFACE, TOKEN ?)",
    [os.environ["HF_TOKEN"]]
)

# Build March feature frame
# client_hash_id is retained because the split is grouped by client.
feature_frame = con.execute("""
    SELECT
        content_hash_id,
        ANY_VALUE(client_hash_id) AS client_hash_id,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        AVG(gsc_avg_position) AS gsc_avg_position,
        SUM(ga4_sessions) AS ga4_sessions,
        SUM(ga4_engaged_sessions) AS ga4_engaged_sessions
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/'
        'fact_content_daily_performance/month=2026-03/*.parquet'
    )
    GROUP BY content_hash_id
""").df()

print("Feature frame shape:", feature_frame.shape)
print("Unique content:", feature_frame["content_hash_id"].nunique())
print("Unique clients:", feature_frame["client_hash_id"].nunique())

# Build March-April decline label
label_data = con.execute("""
    WITH march AS (
        SELECT
            content_hash_id,
            SUM(gsc_impressions) AS march_impressions
        FROM read_parquet(
            'hf://datasets/FlyRank/internship-warehouse/'
            'fact_content_daily_performance/month=2026-03/*.parquet'
        )
        GROUP BY content_hash_id
    ),
    april AS (
        SELECT
            content_hash_id,
            SUM(gsc_impressions) AS april_impressions
        FROM read_parquet(
            'hf://datasets/FlyRank/internship-warehouse/'
            'fact_content_daily_performance/month=2026-04/*.parquet'
        )
        GROUP BY content_hash_id
    )
    SELECT
        march.content_hash_id,
        march.march_impressions,
        april.april_impressions,
        CASE
            WHEN april.april_impressions < march.march_impressions
            THEN 1
            ELSE 0
        END AS is_declining_label
    FROM march
    INNER JOIN april
        ON march.content_hash_id = april.content_hash_id
""").df()

print("\nLabel rows:", len(label_data))

print("\nLabel distribution:")
print(label_data["is_declining_label"].value_counts())

# Merge features and labels
data = feature_frame.merge(
    label_data[["content_hash_id", "is_declining_label"]],
    on="content_hash_id",
    how="inner"
)

# Make row order deterministic before splitting/model training
data = data.sort_values("content_hash_id").reset_index(drop=True)

print("\nRows:", len(data))
print("Clients:", data["client_hash_id"].nunique())

print("\nMissing values:")
print(data.isna().sum())

# Check the special GSC position values
print(
    "\ngsc_avg_position == 0:",
    (data["gsc_avg_position"] == 0).sum()
)

print(
    "gsc_avg_position missing:",
    data["gsc_avg_position"].isna().sum()
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (331437, 7)
Unique content: 331437
Unique clients: 55


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Label rows: 331436

Label distribution:
is_declining_label
0    219469
1    111967
Name: count, dtype: int64

Rows: 331436
Clients: 55

Missing values:
content_hash_id              0
client_hash_id               0
gsc_impressions              0
gsc_clicks                   0
gsc_avg_position        154699
ga4_sessions             70700
ga4_engaged_sessions     70700
is_declining_label           0
dtype: int64

gsc_avg_position == 0: 1434
gsc_avg_position missing: 154699


In [2]:
# ML-08: Client-grouped train/test split

from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        data,
        groups=data["client_hash_id"]
    )
)

train = data.iloc[train_idx].copy()
test = data.iloc[test_idx].copy()

print("Train rows:", len(train))
print("Test rows:", len(test))

print("Train clients:", train["client_hash_id"].nunique())
print("Test clients:", test["client_hash_id"].nunique())

# Explicitly verify that no client appears in both sets
train_clients = set(train["client_hash_id"])
test_clients = set(test["client_hash_id"])

overlap = train_clients.intersection(test_clients)

print("Client overlap:", len(overlap))

print("\nTrain label distribution:")
print(
    train["is_declining_label"]
    .value_counts(normalize=True)
    .sort_index()
    .round(3)
)

print("\nTest label distribution:")
print(
    test["is_declining_label"]
    .value_counts(normalize=True)
    .sort_index()
    .round(3)
)

Train rows: 300879
Test rows: 30557
Train clients: 44
Test clients: 11
Client overlap: 0

Train label distribution:
is_declining_label
0    0.661
1    0.339
Name: proportion, dtype: float64

Test label distribution:
is_declining_label
0    0.67
1    0.33
Name: proportion, dtype: float64


In [3]:
# ML-08: Feature preprocessing

features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions"
]

# Calculate medians using the training set only
train_medians = train[features].copy()

# Treat literal 0 in average position as missing
train_medians["gsc_avg_position"] = (
    train_medians["gsc_avg_position"].replace(0, np.nan)
)

train_medians = train_medians.median()

print("Features:", features)
print("\nTraining medians:")
print(train_medians)

def prep(d):
    X = d[features].copy()

    # Zero is not a valid search position
    X["gsc_avg_position"] = (
        X["gsc_avg_position"].replace(0, np.nan)
    )

    # Use training-set medians only
    X = X.fillna(train_medians)

    return X

X_train = prep(train)
X_test = prep(test)

y_train = train["is_declining_label"]
y_test = test["is_declining_label"]

print("\nRemaining missing values in X_train:")
print(X_train.isna().sum())

print("\nRemaining missing values in X_test:")
print(X_test.isna().sum())

Features: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ga4_engaged_sessions']

Training medians:
gsc_impressions         2.00000
gsc_clicks              0.00000
gsc_avg_position        8.61843
ga4_sessions            0.00000
ga4_engaged_sessions    0.00000
dtype: float64

Remaining missing values in X_train:
gsc_impressions         0
gsc_clicks              0
gsc_avg_position        0
ga4_sessions            0
ga4_engaged_sessions    0
dtype: int64

Remaining missing values in X_test:
gsc_impressions         0
gsc_clicks              0
gsc_avg_position        0
ga4_sessions            0
ga4_engaged_sessions    0
dtype: int64


In [4]:
# ML-08: Warehouse baseline rule

# Training-set median of impressions
imp_median = train["gsc_impressions"].median()

# Calculate CTR using training data
ctr_train = (
    train["gsc_clicks"]
    / train["gsc_impressions"].replace(0, np.nan)
).fillna(0)

# Calculate CTR for the test set
ctr_test = (
    test["gsc_clicks"]
    / test["gsc_impressions"].replace(0, np.nan)
).fillna(0)

# Training-set median CTR
ctr_median = ctr_train.median()

print("Training impression median:", imp_median)
print("Training CTR median:", ctr_median)

# Simple binary baseline:
# high-enough impressions AND low/zero CTR
baseline_pred = (
    (test["gsc_impressions"] >= imp_median)
    & (ctr_test <= ctr_median)
).astype(int)

print("\nBaseline predictions:")
print(baseline_pred.value_counts())

print(
    "\nBaseline positive rate:",
    round(baseline_pred.mean(), 3)
)

Training impression median: 2.0
Training CTR median: 0.0

Baseline predictions:
0    20997
1     9560
Name: count, dtype: int64

Baseline positive rate: 0.313


In [5]:
# ML-08: Deterministic Precision@50

from sklearn.metrics import precision_score

def precision_at_k(y_true, scores, k=50, tie_break=None):
    scores = np.asarray(scores, dtype=float)

    if tie_break is not None:
        tie_break = np.asarray(tie_break, dtype=float)

        # Sort by score descending.
        # If scores are tied, use the tiebreaker descending.
        order = np.lexsort((-tie_break, -scores))
    else:
        # Stable ordering for reproducibility when no tiebreaker is supplied
        order = np.argsort(-scores, kind="stable")

    top_k = order[:k]

    return precision_score(
        np.asarray(y_true)[top_k],
        np.ones(len(top_k)),
        zero_division=0
    )

In [6]:
# ML-08: Evaluate warehouse baseline

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

baseline_roc_auc = roc_auc_score(
    y_test,
    baseline_pred
)

baseline_avg_precision = average_precision_score(
    y_test,
    baseline_pred
)

baseline_precision = precision_score(
    y_test,
    baseline_pred,
    zero_division=0
)

baseline_recall = recall_score(
    y_test,
    baseline_pred,
    zero_division=0
)

baseline_f1 = f1_score(
    y_test,
    baseline_pred,
    zero_division=0
)

baseline_precision_at_50 = precision_at_k(
    y_test,
    baseline_pred,
    k=50,
    tie_break=test["gsc_impressions"].values
)

print("Warehouse baseline performance")
print("--------------------------------")
print("ROC-AUC:", round(baseline_roc_auc, 3))
print("Average Precision:", round(baseline_avg_precision, 3))
print("Precision:", round(baseline_precision, 3))
print("Recall:", round(baseline_recall, 3))
print("F1:", round(baseline_f1, 3))
print("Precision@50:", round(baseline_precision_at_50, 3))

Warehouse baseline performance
--------------------------------
ROC-AUC: 0.705
Average Precision: 0.501
Precision: 0.62
Recall: 0.587
F1: 0.603
Precision@50: 0.68


In [28]:
# ML-08: Train Logistic Regression

from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train, y_train)

logistic_pred = lr_model.predict(X_test)
logistic_proba = lr_model.predict_proba(X_test)[:, 1]

print("Logistic regression trained successfully.")
print("Predicted positives:", int(logistic_pred.sum()))
print("Predicted positive rate:", round(logistic_pred.mean(), 3))

Logistic regression trained successfully.
Predicted positives: 1986
Predicted positive rate: 0.065


In [21]:
# ML-08: Fairly preprocessed Logistic Regression

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression

def log_transform(d):
    X = d[features].copy()

    # Treat zero position as missing
    X["gsc_avg_position"] = X["gsc_avg_position"].replace(0, np.nan)

    # Use training-set medians for missing values
    X = X.fillna(train_medians)

    # Log-transform heavily right-skewed count features
    for col in [
        "gsc_impressions",
        "gsc_clicks",
        "ga4_sessions",
        "ga4_engaged_sessions"
    ]:
        X[col] = np.log1p(X[col])

    return X

X_train_scaled, X_test_scaled = log_transform(train), log_transform(test)

lr_scaled = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

lr_scaled.fit(X_train_scaled, y_train)

lr_scaled_proba = lr_scaled.predict_proba(X_test_scaled)[:, 1]
lr_scaled_pred = lr_scaled.predict(X_test_scaled)

print("Scaled + log-transformed Logistic Regression")
print("ROC-AUC:", round(roc_auc_score(y_test, lr_scaled_proba), 3))
print("Average Precision:", round(
    average_precision_score(y_test, lr_scaled_proba), 3
))
print("Precision:", round(
    precision_score(y_test, lr_scaled_pred, zero_division=0), 3
))
print("Recall:", round(
    recall_score(y_test, lr_scaled_pred, zero_division=0), 3
))
print("F1:", round(
    f1_score(y_test, lr_scaled_pred, zero_division=0), 3
))
print("Precision@50:", round(
    precision_at_k(y_test, lr_scaled_proba, k=50), 3
))

Scaled + log-transformed Logistic Regression
ROC-AUC: 0.845
Average Precision: 0.625
Precision: 0.613
Recall: 0.712
F1: 0.659
Precision@50: 0.68


In [23]:
# ML-08: Decision Tree

from sklearn.tree import DecisionTreeClassifier

tree_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

tree_model.fit(X_train, y_train)

tree_proba = tree_model.predict_proba(X_test)[:, 1]
tree_pred = tree_model.predict(X_test)

print("Decision tree trained successfully.")
print("Predicted positives:", tree_pred.sum())
print("Predicted positive rate:", round(tree_pred.mean(), 3))

Decision tree trained successfully.
Predicted positives: 16252
Predicted positive rate: 0.532


In [24]:
# ML-08: Random Forest

from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_proba = rf_model.predict_proba(X_test)[:, 1]
rf_pred = rf_model.predict(X_test)

print("Random forest trained successfully.")
print("Predicted positives:", rf_pred.sum())
print("Predicted positive rate:", round(rf_pred.mean(), 3))

Random forest trained successfully.
Predicted positives: 16417
Predicted positive rate: 0.537


In [29]:
# ML-08: Evaluate all models on the same test set

def evaluate_model(y_true, pred, proba):
    return {
        "roc_auc": roc_auc_score(y_true, proba),
        "avg_precision": average_precision_score(y_true, proba),
        "precision_at_50": precision_at_k(y_true, proba, k=50),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0)
    }

results = {}

# Warehouse baseline
results["warehouse_baseline"] = {
    "roc_auc": baseline_roc_auc,
    "avg_precision": baseline_avg_precision,
    "precision_at_50": baseline_precision_at_50,
    "precision": baseline_precision,
    "recall": baseline_recall,
    "f1": baseline_f1
}

# Logistic regression
results["logistic_regression"] = evaluate_model(
    y_test,
    logistic_pred,
    logistic_proba
)

# Decision tree
results["decision_tree"] = evaluate_model(
    y_test,
    tree_pred,
    tree_proba
)

# Random forest
results["random_forest"] = evaluate_model(
    y_test,
    rf_pred,
    rf_proba
)

comparison = pd.DataFrame(results).T.round(3)

print(comparison)

                     roc_auc  avg_precision  precision_at_50  precision  \
warehouse_baseline     0.705          0.501             0.68      0.620   
logistic_regression    0.648          0.488             0.62      0.494   
decision_tree          0.838          0.611             0.46      0.601   
random_forest          0.851          0.654             0.74      0.602   

                     recall     f1  
warehouse_baseline    0.587  0.603  
logistic_regression   0.097  0.163  
decision_tree         0.967  0.741  
random_forest         0.978  0.745  


In [30]:
# ML-08: Final comparison table

comparison = comparison[
    [
        "roc_auc",
        "avg_precision",
        "precision_at_50",
        "precision",
        "recall",
        "f1"
    ]
]

display(comparison)

print("\nBest model by ROC-AUC:",
      comparison["roc_auc"].idxmax())

print("Best model by Average Precision:",
      comparison["avg_precision"].idxmax())

print("Best model by Precision@50:",
      comparison["precision_at_50"].idxmax())

print("Best model by F1:",
      comparison["f1"].idxmax())

,roc_auc,avg_precision,precision_at_50,precision,recall,f1
warehouse_baseline,0.705,0.501,0.68,0.620,0.587,0.603
logistic_regression,0.648,0.488,0.62,0.494,0.097,0.163
decision_tree,0.838,0.611,0.46,0.601,0.967,0.741
random_forest,0.851,0.654,0.74,0.602,0.978,0.745



Best model by ROC-AUC: random_forest
Best model by Average Precision: random_forest
Best model by Precision@50: random_forest
Best model by F1: random_forest


In [31]:
# ML-08: Random Forest error analysis

best_model = rf_model

rf_proba = best_model.predict_proba(X_test)[:, 1]
rf_pred = best_model.predict(X_test)

false_pos = test[
    (rf_pred == 1) & (y_test == 0)
].copy()

false_neg = test[
    (rf_pred == 0) & (y_test == 1)
].copy()

print("False positives:", len(false_pos))
print("False negatives:", len(false_neg))

print("\nFalse-positive feature summary:")
display(false_pos[features].describe())

print("\nFalse-negative feature summary:")
display(false_neg[features].describe())

False positives: 6541
False negatives: 223

False-positive feature summary:


,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions
count,6541.000000,6541.000000,6541.000000,6332.000000,6332.000000
mean,1407.594252,4.187739,15.101447,4.082280,0.291061
std,8237.644636,70.949135,14.398551,36.061343,3.001200
min,1.000000,0.000000,0.000000,0.000000,0.000000
25%,21.000000,0.000000,5.554744,0.000000,0.000000
50%,238.000000,0.000000,9.142924,1.000000,0.000000
75%,1147.000000,2.000000,19.968750,3.000000,0.000000
max,617124.000000,5668.000000,99.000000,2730.000000,224.000000



False-negative feature summary:


,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions
count,223.000000,223.000000,223.000000,223.000000,223.000000
mean,10319.753363,59.802691,4.481245,51.955157,4.107623
std,22096.711942,98.872821,1.898012,88.009816,10.131437
min,179.000000,4.000000,1.497410,3.000000,0.000000
25%,1335.500000,13.000000,3.337699,10.000000,0.000000
50%,3675.000000,26.000000,4.082675,22.000000,1.000000
75%,9061.500000,51.000000,5.223640,39.500000,2.000000
max,203497.000000,703.000000,17.241444,684.000000,104.000000


In [32]:
# ML-08: Random Forest feature importance

importances = pd.Series(
    rf_model.feature_importances_,
    index=features
).sort_values(ascending=False)

print("Random Forest feature importance:")
display(importances.to_frame("importance"))

Random Forest feature importance:


,importance
gsc_impressions,0.660708
gsc_avg_position,0.242331
gsc_clicks,0.067865
ga4_sessions,0.028073
ga4_engaged_sessions,0.001023


In [33]:
# ML-08: Final results summary

print("=== ML-08 FINAL RESULTS ===\n")

print("Model comparison:")
display(comparison)

print("\nBest model by ROC-AUC:",
      comparison["roc_auc"].idxmax())

print("Best model by Average Precision:",
      comparison["avg_precision"].idxmax())

print("Best model by Precision@50:",
      comparison["precision_at_50"].idxmax())

print("Best model by F1:",
      comparison["f1"].idxmax())

print("\nRandom Forest feature importance:")
display(importances.to_frame("importance"))

print("\nRandom Forest error analysis:")
print("False positives:", len(false_pos))
print("False negatives:", len(false_neg))

=== ML-08 FINAL RESULTS ===

Model comparison:


,roc_auc,avg_precision,precision_at_50,precision,recall,f1
warehouse_baseline,0.705,0.501,0.68,0.620,0.587,0.603
logistic_regression,0.648,0.488,0.62,0.494,0.097,0.163
decision_tree,0.838,0.611,0.46,0.601,0.967,0.741
random_forest,0.851,0.654,0.74,0.602,0.978,0.745



Best model by ROC-AUC: random_forest
Best model by Average Precision: random_forest
Best model by Precision@50: random_forest
Best model by F1: random_forest

Random Forest feature importance:


,importance
gsc_impressions,0.660708
gsc_avg_position,0.242331
gsc_clicks,0.067865
ga4_sessions,0.028073
ga4_engaged_sessions,0.001023



Random Forest error analysis:
False positives: 6541
False negatives: 223


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.